<a href="https://colab.research.google.com/github/suryansh24-coder/Machine-Learning-Journey/blob/main/Interactive_Dashboard_using_Streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from datetime import datetime, timedelta
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
import pickle
import os
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

st.set_page_config(page_title="Machine Health Monitoring Dashboard", layout="wide")

@st.cache_data
def load_initial_data():
    np.random.seed(42)
    timestamps = [datetime.now() - timedelta(hours=i) for i in range(100, 0, -1)]
    data = {
        "Timestamp": timestamps,
        "Temperature": np.random.normal(75, 5, 100),
        "Vibration": np.random.normal(1.2, 0.2, 100),
        "Pressure": np.random.normal(50, 4, 100)
    }
    df = pd.DataFrame(data)
    df["Anomaly"] = (df["Temperature" ] > 86) | (df["Vibration" ] > 1.6)
    df["Failure_Label"] = ((df["Temperature"] * 0.4 + df["Vibration"] * 15 + df["Pressure"] * 0.2 + np.random.normal(0, 5, 100)) > 65).astype(int)
    return df

if "history_df" not in st.session_state:
    st.session_state.history_df = load_initial_data()

df = st.session_state.history_df

features = ["Temperature", "Vibration", "Pressure"]
X = df[features]
y = df["Failure_Label"]

model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X, y)

model_filename = "machine_health_model.pkl"
with open(model_filename, "wb") as f:
    pickle.dump(model, f)

probabilities = model.predict_proba(X)[:, 1] * 100
df["Failure_Prediction_Pct"] = probabilities

st.sidebar.title("Dashboard Settings")

st.sidebar.subheader("Email Notification Config")
enable_email = st.sidebar.checkbox("Enable Email Alerts (Risk > 80%)", value=False)
smtp_server = st.sidebar.text_input("SMTP Server", "smtp.gmail.com")
smtp_port = st.sidebar.number_input("SMTP Port", value=587)
sender_email = st.sidebar.text_input("Sender Email")
sender_password = st.sidebar.text_input("Sender App Password", type="password")
receiver_email = st.sidebar.text_input("Recipient Email")

def send_email_alert(risk_pct, temp, vib, press):
    if not sender_email or not sender_password or not receiver_email:
        st.sidebar.warning("Email config incomplete. Please fill out all fields.")
        return

    subject = "⚠️ CRITICAL: Machine Failure Risk Alert!"
    body = f"""\
    Emergency Alert: Machine failure risk has exceeded safe parameters.

    Current Failure Probability: {risk_pct:.1f}%
    Temperature: {temp:.2f} °C
    Vibration: {vib:.2f} mm/s
    Pressure: {press:.2f} PSI

    Please inspect the machinery immediately.
    """

    msg = MIMEMultipart()
    msg["From"] = sender_email
    msg["To"] = receiver_email
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "plain"))

    try:
        server = smtplib.SMTP(smtp_server, smtp_port)
        server.starttls()
        server.login(sender_email, sender_password)
        server.sendmail(sender_email, receiver_email, msg.as_string())
        server.quit()
        st.sidebar.success("Alert Email sent successfully!")
    except Exception as e:
        st.sidebar.error(f"Failed to send email: {e}")

st.sidebar.subheader("Anomaly Thresholds")
temp_threshold = st.sidebar.slider("Max Temperature Limit (°C)", 60.0, 100.0, 86.0)
vib_threshold = st.sidebar.slider("Max Vibration Limit (mm/s)", 0.5, 3.0, 1.6)
press_threshold = st.sidebar.slider("Max Pressure Limit (PSI)", 30.0, 70.0, 58.0)

df["Anomaly"] = (df["Temperature"] > temp_threshold) | (df["Vibration"] > vib_threshold) | (df["Pressure"] > press_threshold)

st.title("⚙️ Machine Health Monitoring Dashboard (ML Powered)")

latest_row = df.iloc[-1]
temp_alert = latest_row["Temperature"] > temp_threshold
vib_alert = latest_row["Vibration"] > vib_threshold
press_alert = latest_row["Pressure"] > press_threshold
has_anomaly = latest_row["Anomaly"]

col1, col2, col3, col4 = st.columns(4)
with col1:
    st.metric("Temperature (°C)", f"{latest_row['Temperature']:.2f}", delta="Over Limit" if temp_alert else "Normal", delta_color="inverse" if temp_alert else "normal")
with col2:
    st.metric("Vibration (mm/s)", f"{latest_row['Vibration']:.2f}", delta="Over Limit" if vib_alert else "Normal", delta_color="inverse" if vib_alert else "normal")
with col3:
    st.metric("Pressure (PSI)", f"{latest_row['Pressure']:.2f}", delta="Over Limit" if press_alert else "Normal", delta_color="inverse" if press_alert else "normal")
with col4:
    current_X = pd.DataFrame([[latest_row["Temperature"], latest_row["Vibration"], latest_row["Pressure"]]], columns=features)
    failure_risk = model.predict_proba(current_X)[0, 1] * 100
    st.metric("ML Failure Risk", f"{failure_risk:.1f}%", delta="Warning" if failure_risk > 50 else "Safe", delta_color="inverse" if failure_risk > 50 else "normal")

st.subheader("Alerts & Status")
if has_anomaly or failure_risk > 50:
    st.error("🚨 CRITICAL ALERT: High anomaly detection and ML failure probability calculated for this machine!")
else:
    st.success("✅ System Health Status: Healthy")

tab1, tab2, tab3, tab4 = st.tabs(["Trends", "Long-term Performance Analysis", "Sensor Analytics", "Historical Logs"])

with tab1:
    chart_metric = st.selectbox("Select Metric to Visualize", ["Temperature", "Vibration", "Pressure"])
    fig = px.line(df, x="Timestamp", y=chart_metric, title=f"{chart_metric} Trend Analysis")
    fig.add_scatter(x=df[df["Anomaly"] == True]["Timestamp"], y=df[df["Anomaly"] == True][chart_metric], mode='markers', name='Anomalies', marker=dict(color='red', size=10))
    st.plotly_chart(fig, use_container_width=True)

with tab2:
    st.subheader("Long-Term Operational Performance Trends")
    df_sorted = df.sort_values("Timestamp")
    df_sorted["Date"] = pd.to_datetime(df_sorted["Timestamp"]).dt.date
    daily_summary = df_sorted.groupby("Date").agg(
        Avg_Temp=("Temperature", "mean"),
        Max_Temp=("Temperature", "max"),
        Avg_Vib=("Vibration", "mean"),
        Max_Vib=("Vibration", "max"),
        Avg_Press=("Pressure", "mean"),
        Max_Press=("Pressure", "max"),
        Total_Anomalies=("Anomaly", "sum"),
        Avg_Failure_Risk=("Failure_Prediction_Pct", "mean")
    ).reset_index()

    perf_metric = st.selectbox("Select Long-Term Performance Indicator", ["Average Temperature & Max Limits", "Average Vibration Profile", "System Wear Indicator"])
    if perf_metric == "Average Temperature & Max Limits":
        fig_perf = px.line(daily_summary, x="Date", y=["Avg_Temp", "Max_Temp"], title="Daily Temperature Moving Performance Profile", markers=True)
        st.plotly_chart(fig_perf, use_container_width=True)
    elif perf_metric == "Average Vibration Profile":
        fig_perf = px.bar(daily_summary, x="Date", y="Max_Vib", color="Avg_Vib", title="Peak Vibration Fluctuations & Daily Average Intensity", color_continuous_scale="Viridis")
        st.plotly_chart(fig_perf, use_container_width=True)
    else:
        fig_perf = px.area(daily_summary, x="Date", y="Avg_Failure_Risk", title="Estimated Wear Curve over Long-Term Operation Timeframe")
        st.plotly_chart(fig_perf, use_container_width=True)
    st.write("Daily Performance Aggregations")
    st.dataframe(daily_summary)

with tab3:
    col_sa1, col_sa2 = st.columns(2)
    with col_sa1:
        st.subheader("Sensor Correlation Matrix")
        corr = df[["Temperature", "Vibration", "Pressure", "Failure_Prediction_Pct"]].corr()
        fig_corr = px.imshow(corr, text_auto=True, color_continuous_scale='RdBu_r', title="Feature Correlation Coefficients")
        st.plotly_chart(fig_corr, use_container_width=True)
    with col_sa2:
        st.subheader("ML Model Feature Importance")
        importances = model.feature_importances_
        importance_df = pd.DataFrame({
            "Feature": features,
            "Importance": importances
        }).sort_values(by="Importance", ascending=True)
        fig_imp = px.bar(
            importance_df,
            x="Importance",
            y="Feature",
            orientation="h",
            title="Random Forest Feature Contribution to Failure Prediction",
            color="Importance",
            color_continuous_scale="Blues"
        )
        st.plotly_chart(fig_imp, use_container_width=True)

    col_sa3, col_sa4 = st.columns(2)
    with col_sa3:
        st.subheader("Model Evaluation: Confusion Matrix")
        predictions = model.predict(X)
        cm = confusion_matrix(y, predictions)
        fig_cm = px.imshow(
            cm,
            text_auto=True,
            labels=dict(x="Predicted Label", y="True Label"),
            x=["No Failure", "Failure"],
            y=["No Failure", "Failure"],
            color_continuous_scale="Greens"
        )
        st.plotly_chart(fig_cm, use_container_width=True)
    with col_sa4:
        st.subheader("Model Validation: Predicted Risk vs Actual Logged Failures")
        val_df = df.copy()
        val_df["Actual Label"] = val_df["Failure_Label"].map({0: "No Failure Logged (0)", 1: "Actual Failure Logged (1)"})
        fig_val = px.box(
            val_df,
            x="Actual Label",
            y="Failure_Prediction_Pct",
            color="Actual Label",
            title="Distribution of Predicted Failure Risk (%) grouped by Actual Class Label",
            points="all",
            labels={"Failure_Prediction_Pct": "Predicted Failure Probability (%)"}
        )
        st.plotly_chart(fig_val, use_container_width=True)

    st.subheader("Distribution Plots")
    dist_metric = st.selectbox("Select Distribution", ["Temperature", "Vibration", "Pressure"])
    fig_dist = px.histogram(df, x=dist_metric, marginal="box", title=f"{dist_metric} Distribution Profile")
    st.plotly_chart(fig_dist, use_container_width=True)

with tab4:
    col_dl1, col_dl2 = st.columns([3, 1])
    with col_dl1:
        st.write("Full Recorded Run Profile Log")
        if os.path.exists(model_filename):
            with open(model_filename, "rb") as f:
                st.download_button("Download Trained ML Model (.pkl)", data=f, file_name=model_filename, mime="application/octet-stream")
    with col_dl2:
        csv_data = df.to_csv(index=False).encode('utf-8')
        st.download_button("Download History Log CSV", data=csv_data, file_name="machine_telemetry_history.csv", mime="text/csv")
    st.dataframe(df.sort_values(by="Timestamp", ascending=False))

st.subheader("Add Current Measurement (Simulation)")
with st.form("new_measurement"):
    new_temp = st.number_input("Temperature", value=75.0)
    new_vib = st.number_input("Vibration", value=1.2)
    new_press = st.number_input("Pressure", value=50.0)
    submit = st.form_submit_button("Record Measurement")

    if submit:
        new_anomaly = (new_temp > temp_threshold) or (new_vib > vib_threshold) or (new_press > press_threshold)
        new_risk_label = int((new_temp * 0.4 + new_vib * 15 + new_press * 0.2 + np.random.normal(0, 5)) > 65)
        new_data = pd.DataFrame([{
            "Timestamp": datetime.now(),
            "Temperature": new_temp,
            "Vibration": new_vib,
            "Pressure": new_press,
            "Anomaly": new_anomaly,
            "Failure_Label": new_risk_label
        }])

        pred_input = pd.DataFrame([[new_temp, new_vib, new_press]], columns=features)
        new_failure_risk = model.predict_proba(pred_input)[0, 1] * 100

        if enable_email and new_failure_risk > 80.0:
            send_email_alert(new_failure_risk, new_temp, new_vib, new_press)

        st.session_state.history_df = pd.concat([st.session_state.history_df, new_data], ignore_index=True)
        st.success("Measurement logged successfully!")
        st.rerun()


Overwriting app.py


In [5]:
!pip install -q streamlit pyngrok
!nohup streamlit run app.py --server.port 8501 &


nohup: appending output to 'nohup.out'


In [16]:
!nohup npx localtunnel --port 8501 > localtunnel.log 2>&1 &
import time
time.sleep(3)
with open('localtunnel.log', 'r') as f:
    print(f.read())

your url is: https://plain-mirrors-drop.loca.lt



In [17]:
import subprocess
import time

subprocess.Popen(["nohup", "streamlit", "run", "app.py", "--server.port", "8501"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

subprocess.Popen("nohup npx localtunnel --port 8501 > localtunnel_new.log 2>&1 &", shell=True)
time.sleep(5)

with open('localtunnel_new.log', 'r') as f:
    print(f.read())

your url is: https://itchy-rats-clap.loca.lt



In [18]:
import subprocess
import time

subprocess.Popen("pkill -f localtunnel", shell=True)
time.sleep(2)

subprocess.Popen("nohup npx localtunnel --port 8501 > localtunnel_refresh.log 2>&1 &", shell=True)
time.sleep(6)

with open('localtunnel_refresh.log', 'r') as f:
    print(f.read())

your url is: https://large-oranges-sneeze.loca.lt



In [19]:
import subprocess
import time

subprocess.Popen("pkill -f streamlit", shell=True)
subprocess.Popen("pkill -f localtunnel", shell=True)
time.sleep(3)

subprocess.Popen(["nohup", "streamlit", "run", "app.py", "--server.port", "8501"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

subprocess.Popen("nohup npx localtunnel --port 8501 > localtunnel_final.log 2>&1 &", shell=True)
time.sleep(6)

with open('localtunnel_final.log', 'r') as f:
    print(f.read())

your url is: https://all-files-know.loca.lt

